# 🛒 台灣即時差價與單位比價系統 — Google Colab 算力處理中心

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/happyfarmer-dot/taiwan-price-comparison/blob/main/colab/update_taiwan_prices.ipynb)

本筆記本讓您使用 **Google Colab 免費雲端 Python 環境** 驅動比價網頁：
1. **步驟 1**：載入 GitHub 線上資料庫 [`data/prices.json`](https://github.com/happyfarmer-dot/taiwan-price-comparison/blob/main/data/prices.json)
2. **步驟 2**：嘗試呼叫公開網購 API（如 PChome 24h）抓最新價格，並自動比對更新「歷史低點 `histLow`」與「單位價錢」
3. **步驟 3**：互動式更新實體門市最新週末特價（全聯、家樂福、好市多、超商等）
4. **步驟 4**：用 Pandas 與 Matplotlib 分析實體店 vs 網購價差與歷史低點分佈
5. **步驟 5**：輸出最新 `prices.json` 並可一鍵自動推送回 GitHub Pages 線上網站！

> 線上比價網頁：<https://happyfarmer-dot.github.io/taiwan-price-comparison/>

In [ ]:
# @title 步驟 1：讀取線上比價資料庫 (data/prices.json)
import base64
import json
from datetime import datetime, timezone
import requests
import pandas as pd

RAW_JSON_URL = "https://raw.githubusercontent.com/happyfarmer-dot/taiwan-price-comparison/main/data/prices.json"

resp = requests.get(RAW_JSON_URL, timeout=15)
resp.raise_for_status()
payload = resp.json()
products = payload["products"]

print(f"成功載入商品總數：{len(products)} 款")
print(f"資料庫最後更新：{payload.get('updatedAt')} ({payload.get('updatedBy')})")

rows = []
for p in products:
    for o in p["offers"]:
        unit_price = round((o["price"] / o["qty"]) * p["displayPer"], 2) if o["qty"] > 0 else None
        rows.append({
            "ID": p["id"],
            "品牌": p["brand"],
            "商品名稱": p["name"],
            "品項標題": o.get("itemTitle", p["name"]),
            "規格": o["specText"],
            "通路": o["storeId"],
            "售價": o["price"],
            "歷史低點": o.get("histLow"),
            "歷史均價": o.get("histAvg"),
            "單位價": unit_price,
            "促銷": o.get("promo", "")
        })

df_preview = pd.DataFrame(rows)
df_preview.head(15)

In [ ]:
# @title 步驟 2：Colab 自動查價與歷史低點 (histLow) 更新引擎
# 說明：嘗試透過 PChome 24h 公開搜尋 API 抓取比對；若雲端節點被限流，會自動保留既有安全基準價。
ENABLE_LIVE_FETCH = True  # @param {type:"boolean"}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*"
}

def fetch_pchome_best_match(keyword, baseline_price):
    url = "https://ecshweb.pchome.com.tw/search/v3.3/all/results"
    r = requests.get(url, params={"q": keyword, "page": 1, "sort": "rnk/dc"}, headers=HEADERS, timeout=8)
    r.raise_for_status()
    data = r.json()
    prods = data.get("prods") or []
    # 只採納合理價格區間內（避免抓取到單包試用包或不相干配件），確保比價規格一致
    candidates = [
        item for item in prods[:8]
        if isinstance(item.get("price"), (int, float))
        and baseline_price * 0.55 <= item["price"] <= baseline_price * 1.45
    ]
    if not candidates:
        return None
    best = min(candidates, key=lambda x: x["price"])
    return {
        "price": int(best["price"]),
        "name": best.get("name", ""),
        "url": f"https://24h.pchome.com.tw/prod/{best['Id']}" if best.get("Id") else ""
    }

updated_count = 0
if ENABLE_LIVE_FETCH:
    for p in products:
        for o in p["offers"]:
            if o["storeId"] == "pchome":
                kw = p.get("searchKey") or f"{p['brand']} {p['name']}"
                try:
                    hit = fetch_pchome_best_match(kw, o["price"])
                    if hit:
                        old_p = o["price"]
                        o["price"] = hit["price"]
                        if hit["url"]:
                            o["url"] = hit["url"]
                        if hit["price"] < o.get("histLow", hit["price"]):
                            o["histLow"] = hit["price"]
                            o["promo"] = "Colab偵測創歷史新低!"
                        updated_count += 1
                        print(f"✅ [{p['brand']} {p['name']}] PChome 即時價：NT$ {old_p} -> NT$ {hit['price']} ({hit['name'][:28]}...)")
                    else:
                        print(f"ℹ️ [{p['brand']} {p['name']}] PChome 無同規格價位匹配，維持 NT$ {o['price']}")
                except Exception as exc:
                    print(f"⚠️ [{p['brand']} {p['name']}] PChome 請求略過（{type(exc).__name__}），保留 NT$ {o['price']}")
else:
    print("已跳過即時網路抓價，僅執行歷史低點與單位價檢查。")

# 確保每筆品項都具備 histLow / histAvg 基準欄位
for p in products:
    for o in p["offers"]:
        if not o.get("histLow"):
            o["histLow"] = int(round(o["price"] * 0.94))
        elif o["price"] < o["histLow"]:
            o["histLow"] = o["price"]
        if not o.get("histAvg"):
            o["histAvg"] = int(round(o["price"] * 1.05))

print(f"\n步驟 2 完成：共成功即時校準 {updated_count} 筆 PChome 報價。")

In [ ]:
# @title 步驟 2B：萬用商品關鍵字即時查價（不限內建 12 款，可搜尋任何商品）
SEARCH_KEYWORD = "AirPods"  # @param {type:"string"}
MAX_RESULTS = 10  # @param {type:"integer"}
ADD_CHEAPEST_TO_JSON = False  # @param {type:"boolean"}

if SEARCH_KEYWORD.strip():
    r = requests.get(
        "https://ecshweb.pchome.com.tw/search/v3.3/all/results",
        params={"q": SEARCH_KEYWORD.strip(), "page": 1, "sort": "prc/ac"},
        headers=HEADERS,
        timeout=10
    )
    r.raise_for_status()
    prods = [p for p in (r.json().get("prods") or []) if isinstance(p.get("price"), (int, float)) and p["price"] > 0]
    live_rows = []
    for item in prods[:MAX_RESULTS]:
        live_rows.append({
            "商品名稱": item.get("name", ""),
            "即時售價(NT$)": int(item["price"]),
            "原價": item.get("originPrice"),
            "購買連結": f"https://24h.pchome.com.tw/prod/{item['Id']}" if item.get("Id") else ""
        })
    df_live = pd.DataFrame(live_rows)
    print(f"🔎「{SEARCH_KEYWORD}」PChome 24h 即時查價結果（共 {len(df_live)} 筆，依價錢由低到高）：")
    display(df_live)

    if ADD_CHEAPEST_TO_JSON and live_rows:
        best = live_rows[0]
        new_id = f"kw_{len(products) + 1}"
        products.append({
            "id": new_id,
            "brand": "Colab即時",
            "name": SEARCH_KEYWORD.strip(),
            "shortLabel": f"⚡ {SEARCH_KEYWORD.strip()}",
            "searchKey": SEARCH_KEYWORD.strip(),
            "category": "即時查價",
            "baseUnit": "件",
            "displayPer": 1,
            "offers": [{
                "storeId": "pchome",
                "itemTitle": best["商品名稱"],
                "price": best["即時售價(NT$)"],
                "histLow": best["即時售價(NT$)"],
                "histAvg": int(round(best["即時售價(NT$)"] * 1.05)),
                "qty": 1,
                "specText": "即時查價",
                "promo": "Colab萬用查價",
                "shipping": "網購配送",
                "url": best["購買連結"]
            }]
        })
        print(f"✅ 已將最低價品項加入待匯出清單（ID: {new_id}）！")
else:
    print("請輸入 SEARCH_KEYWORD 即可查詢任何商品的即時價錢。")

In [ ]:
# @title 步驟 3：（選用）在 Colab 手動調整任一通路最新門市特價
TARGET_PRODUCT_ID = "p1"  # @param ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8", "p9", "p10", "p11", "p12"]
TARGET_STORE_ID = "pxmart"  # @param ["pxmart", "carrefour", "rtmart", "amart", "costco", "simplemart", "711", "familymart", "hilife", "okmart", "poya", "watsons", "cosmed", "showba", "momo", "pchome", "shopee", "coupang", "yahoo"]
NEW_PRICE = 0  # @param {type:"integer"}
NEW_PROMO_TAG = "Colab 手動校準特價"  # @param {type:"string"}

if NEW_PRICE > 0:
    applied = False
    for p in products:
        if p["id"] == TARGET_PRODUCT_ID:
            for o in p["offers"]:
                if o["storeId"] == TARGET_STORE_ID:
                    old = o["price"]
                    o["price"] = NEW_PRICE
                    o["promo"] = NEW_PROMO_TAG
                    o["histLow"] = min(o.get("histLow", NEW_PRICE), NEW_PRICE)
                    applied = True
                    print(f"✅ 已更新 {p['brand']} {p['name']} ({TARGET_STORE_ID})：NT$ {old} -> NT$ {NEW_PRICE}")
    if not applied:
        print(f"⚠️ 找不到 {TARGET_PRODUCT_ID} 在 {TARGET_STORE_ID} 的報價紀錄。")
else:
    print("NEW_PRICE = 0，略過手動調價（保留目前數據）。")

In [ ]:
# @title 步驟 4：Colab 分析報表 — 實體門市 vs 網購平台最低單位價差對比
import matplotlib.pyplot as plt

PHYSICAL_IDS = {
    "pxmart", "carrefour", "rtmart", "amart", "costco", "simplemart",
    "711", "familymart", "hilife", "okmart", "poya", "watsons", "cosmed", "showba"
}

summary_rows = []
for p in products:
    phys_offers = [o for o in p["offers"] if o["storeId"] in PHYSICAL_IDS]
    onl_offers = [o for o in p["offers"] if o["storeId"] not in PHYSICAL_IDS]

    phys_best = min(phys_offers, key=lambda o: (o["price"] / o["qty"]) * p["displayPer"]) if phys_offers else None
    onl_best = min(onl_offers, key=lambda o: (o["price"] / o["qty"]) * p["displayPer"]) if onl_offers else None

    phys_unit = round((phys_best["price"] / phys_best["qty"]) * p["displayPer"], 2) if phys_best else None
    onl_unit = round((onl_best["price"] / onl_best["qty"]) * p["displayPer"], 2) if onl_best else None

    winner = "實體勝出" if (phys_unit and (not onl_unit or phys_unit <= onl_unit)) else "網購勝出"
    summary_rows.append({
        "ID": p["id"],
        "品牌商品": f"{p['brand']} {p['name']}",
        "基準": f"每{p['displayPer']}{p['baseUnit']}",
        "實體最低通路": phys_best["storeId"] if phys_best else "-",
        "實體最低單價": phys_unit,
        "網購最低通路": onl_best["storeId"] if onl_best else "-",
        "網購最低單價": onl_unit,
        "划算通路": winner
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

# 繪製各商品實體最低單價 vs 網購最低單價對比條狀圖 (使用商品 ID 避免 Colab 預設缺少繁中字型變亂碼)
plot_df = df_summary.dropna(subset=["實體最低單價", "網購最低單價"]).copy()
plot_df["DiffRatioPct"] = ((plot_df["網購最低單價"] - plot_df["實體最低單價"]) / plot_df["實體最低單價"]) * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#059669" if v >= 0 else "#0284c7" for v in plot_df["DiffRatioPct"]]
ax.barh(plot_df["ID"], plot_df["DiffRatioPct"], color=colors)
ax.axvline(0, color="#6b7280", linewidth=1)
ax.set_xlabel("Online vs Physical Unit Price Diff (%)  [>0: Physical Cheaper | <0: Online Cheaper]")
ax.set_ylabel("Product ID")
ax.set_title("Physical vs Online Unit Price Spread by Product")
plt.tight_layout()
plt.show()

In [ ]:
# @title 步驟 5：匯出最新 data/prices.json 並可選推送回 GitHub 更新網站
GITHUB_OWNER = "happyfarmer-dot"  # @param {type:"string"}
GITHUB_REPO = "taiwan-price-comparison"  # @param {type:"string"}
GITHUB_BRANCH = "main"  # @param {type:"string"}
DOWNLOAD_LOCAL_JSON = True  # @param {type:"boolean"}
PUSH_TO_GITHUB = False  # @param {type:"boolean"}

payload["updatedAt"] = datetime.now(timezone.utc).isoformat()
payload["updatedBy"] = "Google Colab 自動運算與更新"
payload["products"] = products

out_text = json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
with open("prices.json", "w", encoding="utf-8") as f:
    f.write(out_text)
print("✅ 已產生 /content/prices.json")

if DOWNLOAD_LOCAL_JSON:
    try:
        from google.colab import files
        files.download("prices.json")
    except Exception:
        print("ℹ️ 非 Colab 互動環境，請於目前目錄查看 prices.json。")

if PUSH_TO_GITHUB:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None

    if not token:
        from getpass import getpass
        token = getpass("請貼上具備 repo 寫入權限的 GitHub Personal Access Token (輸入不會顯示)：")

    api_url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/data/prices.json"
    gh_headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }
    meta = requests.get(api_url, params={"ref": GITHUB_BRANCH}, headers=gh_headers, timeout=15)
    meta.raise_for_status()
    current_sha = meta.json()["sha"]

    commit_resp = requests.put(
        api_url,
        headers=gh_headers,
        json={
            "message": f"Update data/prices.json via Google Colab ({payload['updatedAt']})",
            "content": base64.b64encode(out_text.encode("utf-8")).decode("ascii"),
            "sha": current_sha,
            "branch": GITHUB_BRANCH
        },
        timeout=20
    )
    commit_resp.raise_for_status()
    commit_url = commit_resp.json()["commit"]["html_url"]
    print(f"🚀 已成功推送更新至 GitHub！Commit：{commit_url}")
    print(f"🌐 約 30 秒後線上網頁即同步更新：https://{GITHUB_OWNER}.github.io/{GITHUB_REPO}/")